# AeroVision LK — Autoencoder Anomaly Detection: Evaluation

**Phase 3 | Threshold Calibration & Evaluation**  
**Model:** ConvAutoencoder trained on MVTec AD hazelnut (normal only)  
**Goal:** Calibrate anomaly threshold from val split, evaluate on test split, compare threshold strategies

---
**Outputs:**
- `reports/figures/eval_error_histogram.png` — reconstruction error distributions
- `reports/figures/eval_pr_curve.png` — precision-recall curve with operating points
- `reports/figures/eval_heatmap_grid.png` — normal vs anomaly heatmap overlays
- `reports/eval_metrics.csv` — threshold comparison table

---
## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path

import torch
from IPython.display import Image as IPImage

ROOT        = Path('..')
FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH  = ROOT / 'models' / 'best_autoencoder.pt'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device:     {device}')
if device == 'cuda':
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Model path: {MODEL_PATH.resolve()}')
print(f'Model exists: {MODEL_PATH.exists()}')

---
## 2. Load Trained Model

In [ ]:
from src.model import ConvAutoencoder

model = ConvAutoencoder().to(device)
checkpoint = torch.load(MODEL_PATH, map_location=device)

# Handle both raw state-dict and checkpoint-dict formats
state_dict = checkpoint.get('model_state_dict', checkpoint)
model.load_state_dict(state_dict)
model.eval()

print(model)
print()
if 'epoch' in checkpoint:
    print(f'Loaded checkpoint — epoch {checkpoint["epoch"]},  val_loss {checkpoint.get("val_loss", "n/a"):.6f}')

---
## 3. Build DataLoaders

In [ ]:
from src.dataset import get_dataloaders, MVTecDataset

loaders = get_dataloaders(ROOT, batch_size=16, num_workers=0)

val_loader  = loaders['val']
test_loader = loaders['test']

test_dataset = MVTecDataset(ROOT, split='test')

print(f'Val  loader : {len(val_loader.dataset)} images (all normal)')
print(f'Test dataset: {test_dataset}')

---
## 4. Threshold Calibration (Val Split)

The val split contains **only normal images**.  
We fit the reconstruction-error distribution and derive four threshold candidates:

| Variant | Rule |
|---|---|
| `mu+2σ` | mean + 2 × std — permissive, higher recall |
| `mu+3σ` | mean + 3 × std — conservative, higher precision |
| `p95`   | 95th-percentile of val errors |
| `fixed` | 0.005 hardcoded baseline |

In [ ]:
from src.threshold import compute_dynamic_threshold

threshold_dict = compute_dynamic_threshold(model, val_loader, device, k=2.0)

print(f"Val error distribution:")
print(f"  mu    = {threshold_dict['mu']:.6f}")
print(f"  sigma = {threshold_dict['sigma']:.6f}")
print()
print(f"Threshold candidates:")
print(f"  fixed = {threshold_dict['threshold_fixed']:.6f}")
print(f"  mu+2σ = {threshold_dict['threshold_mu2']:.6f}")
print(f"  mu+3σ = {threshold_dict['threshold_mu3']:.6f}")
print(f"  p95   = {threshold_dict['threshold_p95']:.6f}")

---
## 5. Evaluate All Threshold Variants (Test Split)

In [ ]:
from src.threshold import build_comparison_table

df_metrics, test_errors, test_labels = build_comparison_table(
    model, test_loader, device, threshold_dict
)

print(df_metrics.to_string(index=False, float_format='{:.4f}'.format))

# Save to CSV
csv_path = ROOT / 'reports' / 'eval_metrics.csv'
csv_path.parent.mkdir(parents=True, exist_ok=True)
df_metrics.to_csv(csv_path, index=False, float_format='%.4f')
print(f'\nSaved: {csv_path.resolve()}')

---
## 6. Reconstruction Error Histogram

Overlays three distributions:
- **Val normal** — in-distribution; anchors threshold calibration
- **Test normal** — should look similar to val
- **Test anomaly** — should shift right (higher errors)

Vertical lines mark each threshold variant.

In [ ]:
from src.visualize import plot_reconstruction_errors

# Split test errors by label
test_normal_errors  = [e for e, lbl in zip(test_errors, test_labels) if lbl == 0]
test_anomaly_errors = [e for e, lbl in zip(test_errors, test_labels) if lbl == 1]

print(f'Test normal  : {len(test_normal_errors)} images')
print(f'Test anomaly : {len(test_anomaly_errors)} images')

hist_path = FIGURES_DIR / 'eval_error_histogram.png'
plot_reconstruction_errors(
    val_errors          = threshold_dict['all_errors'],
    test_normal_errors  = test_normal_errors,
    test_anomaly_errors = test_anomaly_errors,
    threshold_dict      = threshold_dict,
    save_path           = hist_path,
)
IPImage(str(hist_path))

---
## 7. Precision-Recall Curve

The PR curve summarises detector performance across all possible thresholds.  
Operating points (mu+2σ, mu+3σ, p95) are marked to show their precision/recall trade-off.

In [ ]:
from src.visualize import plot_pr_curve

pr_path = FIGURES_DIR / 'eval_pr_curve.png'
plot_pr_curve(
    test_errors    = test_errors,
    test_labels    = test_labels,
    threshold_dict = threshold_dict,
    save_path      = pr_path,
)
IPImage(str(pr_path))

---
## 8. Error Heatmap Grid

Side-by-side: original image | per-pixel reconstruction error heatmap.  
Top rows = normal images (low uniform error).  
Bottom rows = anomalous images (high localised error at defect regions).

In [ ]:
from src.visualize import plot_heatmap_grid

heatmap_path = FIGURES_DIR / 'eval_heatmap_grid.png'
plot_heatmap_grid(
    model        = model,
    test_dataset = test_dataset,
    device       = device,
    n_normal     = 4,
    n_anomaly    = 4,
    save_path    = heatmap_path,
)
IPImage(str(heatmap_path))

---
## 9. Summary

In [ ]:
import numpy as np

best_row = df_metrics.loc[df_metrics['f1'].idxmax()]
pr_auc   = df_metrics['pr_auc'].iloc[0]  # same for all rows (threshold-independent)

print('=' * 58)
print('  ConvAutoencoder — Evaluation Summary')
print('=' * 58)
print(f'  Dataset   : MVTec AD — hazelnut')
print(f'  Test split: {len(test_errors)} images  '
      f'({len(test_normal_errors)} normal / {len(test_anomaly_errors)} anomaly)')
print()
print('  Threshold Comparison:')
print(f'  {df_metrics.to_string(index=False, float_format="{:.4f}".format)}')
print()
print(f'  Best strategy : {best_row["threshold_type"]}  '
      f'(F1={best_row["f1"]:.4f},  '
      f'P={best_row["precision"]:.4f},  '
      f'R={best_row["recall"]:.4f})')
print(f'  PR-AUC        : {pr_auc:.4f}')
print()
print('  Saved figures:')
print('    eval_error_histogram.png  — error distributions + threshold lines')
print('    eval_pr_curve.png         — PR curve with operating points')
print('    eval_heatmap_grid.png     — original | heatmap pairs')
print('    eval_metrics.csv          — threshold comparison table')
print('=' * 58)